# 03 Prevent Pce Data Prep R

PREVENT data preparation and descriptive analysis (R)

Purpose: prepare the main analytic dataset, generate baseline summaries, risk calculations, calibration summaries, and supporting descriptive outputs.

# Initialize Truveta SDK

In [ ]:
library(readr, warn.conflicts = FALSE)
library(arrow, warn.conflicts = FALSE)
library(magrittr, warn.conflicts = FALSE)
library(stringr, warn.conflicts = FALSE)
library(dplyr, warn.conflicts = FALSE)
library(rlang, warn.conflicts = FALSE)
library(data.table, warn.conflicts = FALSE)
library(lubridate, warn.conflicts = FALSE)
library(tidyr, warn.conflicts = FALSE)
library(truveta.notebook.study)
library(tableone)
library(table1)
library(survival)
library(ggplot2)
con <- create_connection(output_mode = "sparklyr")
study <- get_study(con)
population <- get_population(con, id='<TRUVETA_POPULATION_ID>')
snapshot = get_latest_snapshot(con, population)

# PREVENT Data Preparation

In [ ]:
# Load raw dataset
raw_df<- load_artifacts_data(con, study, "/data/PREVENT_PCE_DATA_INS_01_16_2025.parquet")
raw_df <- raw_df %>% collect()

In [ ]:
# Curate insurance variable 
insurance_dict <- c(
  "3058516" = "Children's Health Insurance Program",
  "3058517" = "Commercial",
  "3058518" = "Indian Health Service",
  "3058519" = "Medicaid",
  "3058520" = "Medicare",
  "3058525" = "Other",
  "3058521" = "Self-Pay",
  "3058522" = "Tricare",
  "3058523" = "Veterans' Health Administration",
  "3058524" = "Worker's Compensation"
)

raw_df$insurance <- insurance_dict[as.character(raw_df$PayerTypeConceptId)]

raw_df <- raw_df %>%
  mutate(
    cat_insurance = case_when(
      insurance %in% c("Commercial") ~ "Private Insurance",
      insurance %in% c("Medicaid", "Medicare", "Tricare") ~ "Public Insurance",
      insurance %in% c("Worker's Compensation", "Other") | is.na(insurance) ~ "Other Insurance",
      TRUE ~ "Unknown"
    )
  )

In [ ]:
# Rename columns in a SparkDataFrame
colnames(raw_df) <- colnames(raw_df) %>%
  gsub("FirstDegreeRelativesCount_Attribute_Value_Numeric", "num_first_degree_relatives_count", .) %>%
  gsub("AddressChangeCountLast60Months_Attribute_Value_Numeric", "num_address_change_60_months", .) %>%
  gsub("HighestEduIndividual_Attribute_Value_Categorical", "cat_highest_edu_individual", .) %>%
  gsub("MedicationNonAdherenceRiskCategory_Attribute_Value_Categorical", "cat_med_nonadherence_risk", .) %>%
  gsub("CurrentAddressOwnOrRent_Attribute_Value_Categorical", "cat_address_own_or_rent", .) %>%
  gsub("CollegePastPresent_Attribute_Value_Categorical", "cat_college_past_present", .) %>%
  gsub("CurrAddrDwellType_Attribute_Value_Categorical", "cat_current_address_dwelling_type", .) %>%
  gsub("HealthManagementNonMotivationRiskCategory_Attribute_Value_Categorical", "cat_health_mgmt_nonmotivation_risk", .) %>%
  gsub("HighestEduHousehold_Attribute_Value_Categorical", "cat_highest_edu_household", .) %>%
  gsub("CurrAddrMedianIncome_Attribute_Value_Numeric", "num_current_address_median_income", .) %>%
  gsub("MedicationAdherenceRate_Attribute_Value_Numeric", "num_med_adherence_rate", .) %>%
  gsub("HealthManagementMotivationLevel_Attribute_Value_Numeric", "num_health_mgmt_motivation_level", .) %>%
  gsub("HealthcareCostRiskCategory_Attribute_Value_Categorical", "cat_healthcare_cost_risk", .)%>%
  gsub("Ethnicity", "ethnicity", .)%>%
  gsub("Race", "race", .)

In [ ]:
# Create outcome for CVD
raw_df$fu_ascvd <- pmin(raw_df$fu_str, raw_df$fu_mi, na.rm = TRUE)
raw_df$fu_cvd <- pmin(raw_df$fu_str, raw_df$fu_mi, raw_df$fu_hf, na.rm = TRUE)

In [ ]:
# Subset the analytical dataframe
variables <- c(
    "index_dtt",
    "age",
    "female",
    "ethnicity",
    "race",
    "scre",
    "sbp",
    "bmi",
    "chol",
    "hdlc",
    "hba1c",
    "acr",
    "diabetes",
    "cursmk",
    "antihtn",
    "statin",
    "event_mi",
    "event_str",
    "event_hf",
    "event_death",
    "ascvd",
    "cvd",
    "fu_mi",
    "fu_str",
    "fu_hf",
    "fu_ascvd",
    "fu_cvd",
    "num_first_degree_relatives_count",
    "num_address_change_60_months",
    "cat_insurance",
    "cat_highest_edu_individual",
    "cat_med_nonadherence_risk",
    "cat_address_own_or_rent",
    "cat_current_address_dwelling_type",
    "cat_health_mgmt_nonmotivation_risk",
    "cat_highest_edu_household",
    "num_current_address_median_income",
    "num_med_adherence_rate",
    "num_health_mgmt_motivation_level",
    "cat_healthcare_cost_risk"
)

analytic_raw_df <- raw_df[ , variables]

In [ ]:
# Recategorize the categorical variable
analytic_raw_df$cat_first_degree_relatives_count <- cut(
  analytic_raw_df$num_first_degree_relatives_count,
  breaks = c(0, 25, 50, 75, 100),
  labels = c("0-25", "26-50", "51-75", "76-100"),
  include.lowest = TRUE,
  right = TRUE
)

analytic_raw_df$cat_address_change_60_months <- cut(
  analytic_raw_df$num_address_change_60_months,
  breaks = c(0, 1, 3, Inf),
  labels = c("0-1", "2-3", "4+"),
  include.lowest = TRUE,
  right = TRUE
)

quantiles <- quantile(analytic_raw_df$num_med_adherence_rate, probs = c(0, 1/3, 2/3, 1), na.rm = TRUE)
quantiles <- c(quantiles[1], quantiles[2]-1 , quantiles[3] + 1, quantiles[4])
analytic_raw_df$cat_med_adherence_rate <- cut(
  analytic_raw_df$num_med_adherence_rate,
  breaks = quantiles,
  labels = c("Low", "Medium", "High"),
  include.lowest = TRUE,
  right = TRUE
)

quantiles <- quantile(analytic_raw_df$num_current_address_median_income, probs = c(0, 1/3, 2/3, 1), na.rm = TRUE)
quantiles <- c(quantiles[1], quantiles[2] - 1, quantiles[3] + 1, quantiles[4])
analytic_raw_df$cat_current_address_median_income <- cut(
  analytic_raw_df$num_current_address_median_income,
  breaks = quantiles,
  labels = c("Low", "Medium", "High"),
  include.lowest = TRUE,
  right = TRUE
)

analytic_raw_df$race <- ifelse(
  analytic_raw_df$race == "White", "White",
  ifelse(analytic_raw_df$race == "Black or African American", "Black",
  ifelse(analytic_raw_df$race == "Asian", "Asian", "Others"))
)

analytic_raw_df$ethnicity <- ifelse(
  analytic_raw_df$ethnicity %in% c("Hispanic or Latino", "Mexican", "Puerto Rican", "Cuban"), 
  "Hispanic or Latino", 
  analytic_raw_df$ethnicity
)

analytic_raw_df$cat_highest_edu_individual <- ifelse(
  analytic_raw_df$cat_highest_edu_individual %in% c("2 year college", "4 year college", "Some college", "Graduate school", "No Information"), 
  "College", 
  analytic_raw_df$cat_highest_edu_individual
)

analytic_raw_df$cat_address_own_or_rent <- ifelse(
  analytic_raw_df$cat_address_own_or_rent %in% c("No Information", "Unknown"), 
  "No Information", 
  analytic_raw_df$cat_address_own_or_rent
)

analytic_raw_df$cat_current_address_dwelling_type <- ifelse(
  analytic_raw_df$cat_current_address_dwelling_type %in% c("Firm or business", "Other type", "Rural route or highway contract", "Unknown type", "Post Office box"), 
  "Others", 
  analytic_raw_df$cat_current_address_dwelling_type
)

analytic_raw_df$cat_highest_edu_household <- ifelse(
  analytic_raw_df$cat_highest_edu_household %in% c("2 year college", "4 year college", "Some college", "Graduate school"), 
  "College", 
  analytic_raw_df$cat_highest_edu_household
)

## Table 1

In [ ]:
# Define the variables to summarize
variables <- c(
    "age", "female", "ethnicity", "race", "scre", "sbp", "bmi", 
    "chol", "hdlc", "hba1c", "acr", "diabetes", "cursmk", 
    "antihtn", "statin", "num_first_degree_relatives_count", 
    "num_address_change_60_months", "cat_insurance", "cat_highest_edu_individual", 
    "cat_med_nonadherence_risk", "cat_address_own_or_rent", 
     "cat_current_address_dwelling_type", 
    "cat_health_mgmt_nonmotivation_risk", "cat_highest_edu_household", 
    "num_current_address_median_income", "num_med_adherence_rate", 
    "num_health_mgmt_motivation_level", "cat_healthcare_cost_risk"
)

# CVD Table
cvd_table <- CreateTableOne(
    vars = variables, 
    strata = "cvd", 
    data = analytic_raw_df, 
    factorVars = c("female", "ethnicity", "race", "diabetes", "cursmk", "antihtn", 
                   "statin", "cat_insurance", "cat_highest_edu_individual", "cat_med_nonadherence_risk",
                   "cat_address_own_or_rent", 
                   "cat_current_address_dwelling_type", "cat_health_mgmt_nonmotivation_risk",
                   "cat_highest_edu_household", "cat_healthcare_cost_risk")
)

# ASCVD Table
ascvd_table <- CreateTableOne(
    vars = variables, 
    strata = "ascvd", 
    data = analytic_raw_df, 
    factorVars = c("female", "ethnicity", "race", "diabetes", "cursmk", "antihtn", 
                   "statin", "cat_insurance", "cat_highest_edu_individual", "cat_med_nonadherence_risk",
                   "cat_address_own_or_rent", 
                   "cat_current_address_dwelling_type", "cat_health_mgmt_nonmotivation_risk",
                   "cat_highest_edu_household", "cat_healthcare_cost_risk")
)

# HF Table
hf_table <- CreateTableOne(
    vars = variables, 
    strata = "event_hf", 
    data = analytic_raw_df, 
    factorVars = c("female", "ethnicity", "race", "diabetes", "cursmk", "antihtn", 
                   "statin", "cat_insurance", "cat_highest_edu_individual", "cat_med_nonadherence_risk",
                   "cat_address_own_or_rent", 
                   "cat_current_address_dwelling_type", "cat_health_mgmt_nonmotivation_risk",
                   "cat_highest_edu_household", "cat_healthcare_cost_risk")
)


In [ ]:
# Transformation for PREVENT
prevent_df <- analytic_raw_df %>%
  mutate(
    year = as.numeric(format(index_dtt, "%Y")),
    egfr = 141.57 * (pmin(scre / 0.7, 1) ^ -0.241) * (pmax(scre / 0.7, 1) ^ -1.2) * (0.993839 ^ age) * 1.012,
    egfr = ifelse(female == 0, 
                  141.57 * (pmin(scre / 0.9, 1) ^ -0.302) * (pmax(scre / 0.9, 1) ^ -1.2) * (0.993839 ^ age),
                  egfr),
    fu_ascvd = pmin(fu_mi, fu_str, na.rm = TRUE),
    fu_cvd = pmin(fu_mi, fu_str, fu_hf, na.rm = TRUE),
    age_raw = age,
    age = (age - 55) / 10,
    nonhdlc = (chol - hdlc) * 0.02586 - 3.5,
    hdlc_raw = hdlc,
    hdlc = (hdlc * 0.02586 - 1.3) / 0.3,
    sbp1 = (pmin(sbp, 110) - 110) / 20,
    sbp2 = (pmax(sbp, 110) - 130) / 20,
    bmi1 = (pmin(bmi, 30) - 25) / 5,
    bmi2 = (pmax(bmi, 30) - 30) / 5,
    egfr1 = (pmin(egfr, 60) - 60) / -15,
    egfr2 = (pmax(egfr, 60) - 90) / -15,
    sbp2treat = sbp2 * antihtn,
    nonhdlctreat = nonhdlc * statin,
    agenonhdlc = age * nonhdlc,
    agehdlc = age * hdlc,
    agesbp2 = age * sbp2,
    agediabetes = age * diabetes,
    agecursmk = age * cursmk,
    agebmi2 = age * bmi2,
    ageegfr1 = age * egfr1,
    cons = 1
  )

## Saving PREVENT data for Aim 2

In [ ]:
path = file.path("/data/Luke_analysis_data/prevent_df")
save_artifacts_data(con, study, prevent_df, path)
#saved_path = file.path("/data/Luke_analysis_data/prevent_df")
#saved_df = load_artifacts_data(con, study, saved_path)
#display_df(saved_df)

# Prevent Risk Calculation

In [ ]:
# Coefficients for CVD
cvd_coef_male <- matrix(c( .7688527703,.0736173987,-.0954430997,-.4347344935,.3362658024,.7692856789,.4386870861,.5378978848,.0164826997,.2888790071,
-.1337348968,-.0475924015,.1502729952,-.0517873988,.0191169009,-.1049477011,-.2251947969,-.0895067006,-.1543702036,-3.031167984), nrow = 20, ncol = 1)
cvd_coef_female <- matrix(c(.7939329147,.0305238999,-.1606857032,-.2394002974,.3600780964,.8667603731,.5360739231,.6045917273,.0433769003,.3151671886,
-.1477655023,-.0663611963,.1197879016,-.0819714963,.0306768995,-.0946348011,-.2705700099,-.0787149966,-.1637805998,-3.307728052), nrow = 20, ncol = 1)

# Coefficients for ASCVD
ascvd_coef_male <- matrix(c(.7099847198,.1658663005,-.1144284979,-.2837212086,.3239977062,.7189596891,.3956972957,.369007498,.0203619003,.2036522031,
-.0865581036,-.0322915986,.1145630032,-.0300005004,.0232746992,-.0927024037,-.2018525004,-.0970527008,-.1217081025,-3.500654936), nrow = 20, ncol = 1)
ascvd_coef_female <- matrix(c(.7198830247,.1176967025,-.1511850059,-.0835357979,.3592852056,.8348584771,.4831078053,.4864619076,.039777901,.2265308946,
-.0592373982,-.0395761989,.0844423026,-.0567838997,.0325691998,-.1035984978,-.241754204,-.0791141987,-.167149201,-3.819974899), nrow = 20, ncol = 1)

# Coefficients for HF
hf_coef_male <- matrix(c(.8972641826,-.6811466217,.3634460866,.9237759709,.5023735762,-.0485840999,.3726929128,.6926916838,.0251826998,
.2980921865,-.0497731008,-.1289200932,-.3040924072,-.140168801,.0068126,-.179777801,-3.946391106), nrow = 17, ncol = 1) 
hf_coef_female <- matrix(c(.8998234868,-.4559771121,.3576504886,1.038346052,.5839160085,-.0072293999,.2997705936,.7451637983,.0557086989,
.3534441888,-.0981511027,-.0946663022,-.3581041098,-.1159453019,-.003878,-.1884288937,-4.310409069), nrow = 17, ncol = 1)

# Covariates for CVD and ASCVD
cvd_cov_male <- as.matrix(subset(prevent_df, subset = female == 0, select = c("age","nonhdlc","hdlc","sbp1","sbp2","diabetes","cursmk","egfr1","egfr2",
"antihtn","statin","sbp2treat","nonhdlctreat","agenonhdlc","agehdlc","agesbp2","agediabetes","agecursmk","ageegfr1","cons")))

cvd_cov_female <- as.matrix(subset(prevent_df, subset = female == 1, select = c("age","nonhdlc","hdlc","sbp1","sbp2","diabetes","cursmk","egfr1","egfr2",
"antihtn","statin","sbp2treat","nonhdlctreat","agenonhdlc","agehdlc","agesbp2","agediabetes","agecursmk","ageegfr1","cons")))

# Covariates for HF
hf_cov_male <- as.matrix(subset(prevent_df, subset = female == 0, select = c("age","sbp1","sbp2","diabetes","cursmk","bmi1","bmi2","egfr1","egfr2",
"antihtn","sbp2treat","agesbp2","agediabetes","agecursmk","agebmi2","ageegfr1","cons")))

hf_cov_female <- as.matrix(subset(prevent_df, subset = female == 1, select = c("age","sbp1","sbp2","diabetes","cursmk","bmi1","bmi2","egfr1","egfr2",
"antihtn","sbp2treat","agesbp2","agediabetes","agecursmk","agebmi2","ageegfr1","cons")))

In [ ]:
# 5 year cvd 
cvd_pred_risk_male <- cvd_cov_male %*% cvd_coef_male
cvd_pred_prob_male_5y <- exp(cvd_pred_risk_male - 0.8581311) / (1 + exp(cvd_pred_risk_male - 0.8581311))
cvd_pred_risk_female <- cvd_cov_female %*% cvd_coef_female
cvd_pred_prob_female_5y <- exp(cvd_pred_risk_female - 0.8581311) / (1 + exp(cvd_pred_risk_female - 0.8581311))

# 10 year cvd 
cvd_pred_risk_male <- cvd_cov_male %*% cvd_coef_male
cvd_pred_prob_male_10y <- exp(cvd_pred_risk_male) / (1 + exp(cvd_pred_risk_male))
cvd_pred_risk_female <- cvd_cov_female %*% cvd_coef_female
cvd_pred_prob_female_10y <- exp(cvd_pred_risk_female) / (1 + exp(cvd_pred_risk_female))

# 5 year ascvd
ascvd_pred_risk_male <- cvd_cov_male %*% ascvd_coef_male
ascvd_pred_prob_male_5y <- exp(ascvd_pred_risk_male-0.8711432) / (1 + exp(ascvd_pred_risk_male-0.8711432))
ascvd_pred_risk_female <- cvd_cov_female %*% ascvd_coef_female
ascvd_pred_prob_female_5y <- exp(ascvd_pred_risk_female-0.8711432) / (1 + exp(ascvd_pred_risk_female-0.8711432))

# 10 year ascvd
ascvd_pred_risk_male <- cvd_cov_male %*% ascvd_coef_male
ascvd_pred_prob_male_10y <- exp(ascvd_pred_risk_male) / (1 + exp(ascvd_pred_risk_male))
ascvd_pred_risk_female <- cvd_cov_female %*% ascvd_coef_female
ascvd_pred_prob_female_10y <- exp(ascvd_pred_risk_female) / (1 + exp(ascvd_pred_risk_female))

# 5 year hf
hf_pred_risk_male <- hf_cov_male %*% hf_coef_male
hf_pred_prob_male_5y <- exp(hf_pred_risk_male-0.9269806) / (1 + exp(hf_pred_risk_male-0.9269806))
hf_pred_risk_female <- hf_cov_female %*% hf_coef_female
hf_pred_prob_female_5y <- exp(hf_pred_risk_female-0.9269806) / (1 + exp(hf_pred_risk_female-0.9269806))

# 10 year hf
hf_pred_risk_male <- hf_cov_male %*% hf_coef_male
hf_pred_prob_male_10y <- exp(hf_pred_risk_male) / (1 + exp(hf_pred_risk_male))
hf_pred_risk_female <- hf_cov_female %*% hf_coef_female
hf_pred_prob_female_10y <- exp(hf_pred_risk_female) / (1 + exp(hf_pred_risk_female))

# Female PREVENT Subdataframe
prevent_df_female <- subset(prevent_df, subset = female == 1)

prevent_df_female$`cvd_predicted_probability_5y` <- cvd_pred_prob_female_5y
prevent_df_female$`ascvd_predicted_probability_5y` <- ascvd_pred_prob_female_5y
prevent_df_female$`hf_predicted_probability_5y` <- hf_pred_prob_female_5y

prevent_df_female$`cvd_predicted_probability_10y` <- cvd_pred_prob_female_10y
prevent_df_female$`ascvd_predicted_probability_10y` <- ascvd_pred_prob_female_10y
prevent_df_female$`hf_predicted_probability_10y` <- hf_pred_prob_female_10y


# Male PREVENT Subdataframe
prevent_df_male <- subset(prevent_df, subset = female == 0)

prevent_df_male$`cvd_predicted_probability_5y` <- cvd_pred_prob_male_5y
prevent_df_male$`ascvd_predicted_probability_5y` <- ascvd_pred_prob_male_5y
prevent_df_male$`hf_predicted_probability_5y` <- hf_pred_prob_male_5y

prevent_df_male$`cvd_predicted_probability_10y` <- cvd_pred_prob_male_10y
prevent_df_male$`ascvd_predicted_probability_10y` <- ascvd_pred_prob_male_10y
prevent_df_male$`hf_predicted_probability_10y` <- hf_pred_prob_male_10y

original_prevent_result <- rbind(prevent_df_female, prevent_df_male)


In [ ]:
display_df(original_prevent_result, 10)

# Save the predicted probability for better practice

In [ ]:
path = file.path("/PREVENT_PCE/Aim_1/original_prevent_result")
save_artifacts_data(con, study, original_prevent_result, path)
saved_path = file.path("/PREVENT_PCE/Aim_1/original_prevent_result")
saved_df = load_artifacts_data(con, study, saved_path)
display_df(saved_df)

# CI for PREVENT

In [ ]:
calculate_overall_concordance <- function(data, time_col, event_col, prediction_col) {
  # Create a survival object
  surv_obj <- Surv(time = data[[time_col]], event = data[[event_col]])
  
  # Fit the Cox proportional hazards model
  cox_model <- coxph(surv_obj ~ data[[prediction_col]], data = data)
  
  # Return the concordance index
  return(cox_model$concordance[6])
}

In [ ]:
# function to calculate CI 
calculate_concordance <- function(data, subgroup_col, subgroup_value, 
                                      time_col, event_col, prediction_col) {
  # Subset the data based on the subgroup
  subset_data <- data[data[[subgroup_col]] == subgroup_value, ]
  
  # Create a survival object for the subset
  surv_obj_subset <- Surv(time = subset_data[[time_col]], 
                          event = subset_data[[event_col]])
  
  # Fit the Cox proportional hazards model
  cox_model <- coxph(surv_obj_subset ~ subset_data[[prediction_col]])
  
  # Return the concordance index
  return(cox_model$concordance[6]) 
}

calculate_and_print_ci <- function(data, categorical_var, ci_function, 
                                   time_col, event_col, prediction_col) {
  # Extract unique categories from the specified variable
  unique_categories <- unique(data[[categorical_var]])
  
  # Loop through each category and calculate CI
  for (category in unique_categories) {
    # Calculate CI using the provided function
    ci <- ci_function(data, categorical_var, category, time_col, event_col, prediction_col)
    
    # Print the result
    cat("Concordance for", category, ":", ci, "\n")
  }
}

### CVD

In [ ]:
calculate_overall_concordance(
    data = original_prevent_result,
    time_col = "fu_cvd", 
    event_col = "cvd", 
    prediction_col = "cvd_predicted_probability_10y")

In [ ]:
categories <- c(
  "female", "ethnicity", "race", "cat_insurance", "cat_highest_edu_individual",
  "cat_med_nonadherence_risk", "cat_address_own_or_rent",
  "cat_current_address_dwelling_type", "cat_health_mgmt_nonmotivation_risk",
  "cat_highest_edu_household", "cat_healthcare_cost_risk", 
   "cat_address_change_60_months", 
  "cat_med_adherence_rate", "cat_current_address_median_income"
)

# Loop through the categories and apply the function
for (var in categories) {
  cat("Calculating concordance for variable:", var, "\n")
  calculate_and_print_ci(
    data = original_prevent_result,
    categorical_var = var,
    ci_function = calculate_concordance,
    time_col = "fu_cvd", 
    event_col = "cvd", 
    prediction_col = "cvd_predicted_probability_10y"
  )
}

### ASCVD

In [ ]:
calculate_overall_concordance(
    data = original_prevent_result,
    time_col = "fu_ascvd", 
    event_col = "ascvd", 
    prediction_col = "ascvd_predicted_probability_10y")

In [ ]:
# Loop through the categories and apply the function
for (var in categories) {
  cat("Calculating concordance for variable:", var, "\n")
  calculate_and_print_ci(
    data = original_prevent_result,
    categorical_var = var,
    ci_function = calculate_concordance,
    time_col = "fu_ascvd", 
    event_col = "ascvd", 
    prediction_col = "ascvd_predicted_probability_10y"
  )
}

### HF

In [ ]:
calculate_overall_concordance(
    data = original_prevent_result,
    time_col = "fu_hf", 
    event_col = "event_hf", 
    prediction_col = "hf_predicted_probability_10y")

In [ ]:
# Loop through the categories and apply the function
for (var in categories) {
  cat("Calculating concordance for variable:", var, "\n")
  calculate_and_print_ci(
    data = original_prevent_result,
    categorical_var = var,
    ci_function = calculate_concordance,
    time_col = "fu_hf", 
    event_col = "event_hf", 
    prediction_col = "hf_predicted_probability_10y"
  )
}

## C-Index (95% CI)： 5 and 10 year PREVENT

In [ ]:
events <- c("cvd", "ascvd", "hf")
time_horizons <- c(5, 10)

categories <- c(
  "female", "ethnicity", "race", "cat_insurance", "cat_highest_edu_individual",
  "cat_med_nonadherence_risk", "cat_address_own_or_rent",
  "cat_current_address_dwelling_type", "cat_health_mgmt_nonmotivation_risk",
  "cat_highest_edu_household", "cat_healthcare_cost_risk", 
   "cat_address_change_60_months", 
  "cat_med_adherence_rate", "cat_current_address_median_income")


# 100 bootstrap, each = original size
nboot <- 100
N <- nrow(original_prevent_result)

# store
results_list <- list()
res_counter <- 1

# loop over event and year
for (event in events) {
  time_col  <- paste0("fu_", event)
  if (event == "hf") {
    event_col <- "event_hf"
  } else {
    event_col <- event
  }
  for (horizon in time_horizons) {
    # pred probs
    pred_col <- paste0(event, "_predicted_probability_", horizon, "y")
    
    set.seed(123)
    for (b in seq_len(nboot)) {
      # resample
      sample_idx <- base::sample(seq_len(N), N, replace = TRUE)
      boot_data  <- original_prevent_result[sample_idx, ]
      
      # for each subgroup category
      for (cat_var in categories) {
        
        # determine unique subgroup values
        unique_subgroups <- unique(boot_data[[cat_var]])
        
        for (subgroup_val in unique_subgroups) {
          
          # get c-index
          c_idx <- calculate_concordance(
            data           = boot_data,
            subgroup_col   = cat_var,
            subgroup_value = subgroup_val,
            time_col       = time_col,
            event_col      = event_col,
            prediction_col = pred_col
          )
          
          # store result
          results_list[[res_counter]] <- data.frame(
            event          = event,
            horizon        = horizon,
            category       = cat_var,
            subgroup_value = subgroup_val,
            replicate      = b,
            c_index        = c_idx
          )
          res_counter <- res_counter + 1
        }
      }
    }
  }
}

# concat results
df_results <- do.call(rbind, results_list)

# summarize
summary_results <- df_results %>%
  group_by(event, horizon, category, subgroup_value) %>%
  summarize(
    mean_c_index = mean(c_index, na.rm = TRUE),
    lower_ci     = quantile(c_index, probs = 0.025, na.rm = TRUE),
    upper_ci     = quantile(c_index, probs = 0.975, na.rm = TRUE),
    .groups      = "drop"
  )

# print
for (i in seq_len(nrow(summary_results))) {
  cat(summary_results$horizon[i], 'year', summary_results$event[i], 
    "| Category:", summary_results$category[i], 
    "| Subgroup:", summary_results$subgroup_value[i], 
    "| Mean C-Index (95%CI):", paste0(round(summary_results$mean_c_index[i], 4), "(", 
    round(summary_results$lower_ci[i], 4), "~", round(summary_results$upper_ci[i], 4), ")"), 
    "\n"
  )
}

## Checking Prevalence

In [ ]:
# Define the function
plot_km_curve <- function(data, time, status, group_var, title = "Kaplan-Meier Curve") {
  # Ensure the input columns exist in the data
  if (!all(c(time, status, group_var) %in% colnames(data))) {
    stop("The specified columns do not exist in the data.")
  }
  
  # Create a survival object
  surv_obj <- Surv(data[[time]], data[[status]])
  
  # Fit the Kaplan-Meier model
  fit <- survfit(surv_obj ~ data[[group_var]], data = data)
  
  # Generate the KM plot
  plot(
    fit,
    col = 1:length(unique(data[[group_var]])),   # Assign different colors for groups
    lty = 1,                                    # Line types
    lwd = 2,                                    # Line widths
    xlab = "Time",                              # Label for x-axis
    ylab = "Survival Probability",              # Label for y-axis
    main = title                                # Plot title
  )
  
  # Add a legend
  legend(
    "bottomleft",
    legend = levels(as.factor(data[[group_var]])), # Group names
    col = 1:length(unique(data[[group_var]])),     # Colors for lines
    lty = 1,                                       # Line types
    lwd = 2,                                       # Line widths
    title = group_var                              # Legend title
  )
}

In [ ]:
plot_km_curve(original_prevent_result, "fu_cvd", "cvd", "cat_insurance", title = "KM Curve by RACE")

# Calibration for PREVENT

## Functions for calibration plots

In [ ]:
calibration_plot <- function(data, pred_prob_col, event_col, time_col, k = 10) {
  # Ensure data has necessary columns
  if (!(all(c(pred_prob_col, event_col, time_col) %in% colnames(data)))) {
    stop("Specified columns do not exist in the data frame")
  }
  
  # Assign stratum based on predicted probabilities
  data$stratum <- cut(data[[pred_prob_col]], 
                      breaks = quantile(data[[pred_prob_col]], probs = seq(0, 1, length.out = k + 1), na.rm = TRUE),
                      include.lowest = TRUE, labels = FALSE)
  
  # Initialize vectors for results
  strata_means <- numeric(k)
  observed_probs <- numeric(k)
  
  # Loop through strata to compute calibration points
  for (i in 1:k) {
    stratum_data <- data[data$stratum == i, ]
    
    # Mean predicted probability for the stratum
    strata_means[i] <- mean(stratum_data[[pred_prob_col]], na.rm = TRUE)
    
    # Fit KM curve and get survival probability at the maximum observed time
    km_fit <- survfit(Surv(stratum_data[[time_col]], stratum_data[[event_col]]) ~ 1)
    observed_probs[i] <- 1 - summary(km_fit, times = max(stratum_data[[time_col]], na.rm = TRUE))$surv
  }
  
  # Create a data frame for plotting
  calibration_df <- data.frame(
    Predicted = strata_means,
    Observed = observed_probs
  )
  
  # Plot the calibration curve
  ggplot(calibration_df, aes(x = Predicted, y = Observed)) +
    geom_point(size = 3) +
    geom_line() +
    geom_abline(intercept = 0, slope = 1, linetype = "dashed", color = "red") +
    scale_x_continuous(limits = c(0, 1)) +
    scale_y_continuous(limits = c(0, 1)) +
    labs(
      x = "Predicted Probability",
      y = "Observed Probability",
      title = "Calibration Plot"
    ) +
    theme_minimal()
}

In [ ]:
stratified_calibration_plot <- function(data, pred_prob_col, event_col, time_col, group_col, k = 10) {
  # Ensure data has necessary columns
  if (!(all(c(pred_prob_col, event_col, time_col, group_col) %in% colnames(data)))) {
    stop("Specified columns do not exist in the data frame")
  }
  
  # Filter groups with more than 100 individuals
  group_counts <- table(data[[group_col]])
  valid_groups <- names(group_counts[group_counts > 100])
  data <- data[data[[group_col]] %in% valid_groups, ]
  
  if (nrow(data) == 0) {
    stop("No groups have more than 100 individuals.")
  }
  
  # Create an empty data frame to store calibration results
  calibration_results <- data.frame()
  
  # Loop through each group
  for (group in unique(data[[group_col]])) {
    group_data <- data[data[[group_col]] == group, ]
    
    # Assign stratum based on predicted probabilities
    group_data$stratum <- cut(group_data[[pred_prob_col]], 
                              breaks = quantile(group_data[[pred_prob_col]], 
                                                probs = seq(0, 1, length.out = k + 1), na.rm = TRUE),
                              include.lowest = TRUE, labels = FALSE)
    
    # Initialize vectors for results
    strata_means <- numeric(k)
    observed_probs <- numeric(k)
    
    # Loop through strata to compute calibration points
    for (i in 1:k) {
      stratum_data <- group_data[group_data$stratum == i, ]
      
      # Mean predicted probability for the stratum
      strata_means[i] <- mean(stratum_data[[pred_prob_col]], na.rm = TRUE)
      
      # Fit KM curve and get survival probability at the maximum observed time
      km_fit <- survfit(Surv(stratum_data[[time_col]], stratum_data[[event_col]]) ~ 1)
      observed_probs[i] <- 1 - summary(km_fit, times = max(stratum_data[[time_col]], na.rm = TRUE))$surv
    }
    
    # Store results for this group
    calibration_results <- rbind(
      calibration_results,
      data.frame(
        Group = group,
        Predicted = strata_means,
        Observed = observed_probs
      )
    )
  }
  
  # Plot the calibration curves
  ggplot(calibration_results, aes(x = Predicted, y = Observed, color = Group)) +
    geom_point(size = 3) +
    geom_line() +
    geom_abline(intercept = 0, slope = 1, linetype = "dashed", color = "red") +
    scale_x_continuous(limits = c(0, 1)) +
    scale_y_continuous(limits = c(0, 1)) +
    labs(
      x = "Predicted Probability",
      y = "Observed Probability",
      color = "Group",
      title = "Calibration Plot by Group"
    ) +
    theme_minimal() +
    theme(legend.position = c(0.8, 0.2)) # Position the legend inside the plot
}


## Functions for Calibration Slope

In [ ]:
overall_calibration_slope <- function(data, pred_prob_col, event_col, time_col, k = 10, time = 10) {
  # Ensure data has necessary columns
  if (!(all(c(pred_prob_col, event_col, time_col) %in% colnames(data)))) {
    stop("Specified columns do not exist in the data frame")
  }

  # Assign stratum based on predicted probabilities
  data$stratum <- cut(data[[pred_prob_col]], 
                      breaks = quantile(data[[pred_prob_col]], 
                                        probs = seq(0, 1, length.out = k + 1), na.rm = TRUE),
                      include.lowest = TRUE, labels = FALSE)

  # Initialize vectors for results
  strata_means <- numeric(k)
  observed_probs <- numeric(k)

  # Loop through strata to compute calibration points
  for (i in 1:k) {
    stratum_data <- data[data$stratum == i, ]

    # Mean predicted probability for the stratum
    strata_means[i] <- mean(stratum_data[[pred_prob_col]], na.rm = TRUE)

    # Fit KM curve and get survival probability at the specified time
    km_fit <- survfit(Surv(stratum_data[[time_col]], stratum_data[[event_col]]) ~ 1)
    observed_probs[i] <- 1 - summary(km_fit, times = time)$surv
  }

  # Fit a linear model for calibration slope and intercept
  lm_fit <- lm(observed_probs ~ strata_means)
  calibration_slope <- coef(lm_fit)[2]
  calibration_intercept <- coef(lm_fit)[1]

  # Return results as a data frame
  return(data.frame(
    Calibration_Slope = calibration_slope,
    Calibration_Intercept = calibration_intercept
  ))
}


In [ ]:
stratified_calibration_slope <- function(data, pred_prob_col, event_col, time_col, group_col, k = 20, times = 10) {
  # Ensure data has necessary columns
  if (!(all(c(pred_prob_col, event_col, time_col) %in% colnames(data)))) {
    stop("Specified columns do not exist in the data frame")
  }

  # Create an empty data frame to store calibration results
  calibration_results <- data.frame()

  # Loop through each group
  for (group in unique(data[[group_col]])) {
    group_data <- data[data[[group_col]] == group, ]

    # Assign stratum based on predicted probabilities
    group_data$stratum <- cut(group_data[[pred_prob_col]], 
                              breaks = quantile(group_data[[pred_prob_col]], 
                                                probs = seq(0, 1, length.out = k + 1), na.rm = TRUE),
                              include.lowest = TRUE, labels = FALSE)

    # Initialize vectors for results
    strata_means <- numeric(k)
    observed_probs <- numeric(k)

    # Loop through strata to compute calibration points
    for (i in 1:k) {
      stratum_data <- group_data[group_data$stratum == i, ]

      # Mean predicted probability for the stratum
      strata_means[i] <- mean(stratum_data[[pred_prob_col]], na.rm = TRUE)

      # Fit KM curve and get survival probability at the specified time
      km_fit <- survfit(Surv(stratum_data[[time_col]], stratum_data[[event_col]]) ~ 1)
      observed_probs[i] <- 1 - summary(km_fit, times = times)$surv
    }

    # Fit a linear model for calibration slope and intercept
    lm_fit <- lm(observed_probs ~ strata_means)
    calibration_slope <- coef(lm_fit)[2]
    calibration_intercept <- coef(lm_fit)[1]

    # Store results for this group
    calibration_results <- rbind(
      calibration_results,
      data.frame(
        Group = group,
        Calibration_Slope = calibration_slope,
        Calibration_Intercept = calibration_intercept
      )
    )
  }

  return(calibration_results)
}


## 10 year Calibration 

### CVD

In [ ]:
overall_calibration_slope(
    original_prevent_result,
    "cvd_predicted_probability_10y",
    "cvd",
    "fu_cvd",
    k = 20,
    time = 10
  )

In [ ]:
# Initialize an empty list to store results
results_list <- list()

# Iterate over each category
for (category in categories) {
  # Calculate the stratified calibration slope for the current category
  result_df <- stratified_calibration_slope(
    original_prevent_result,
    "cvd_predicted_probability_10y",
    "cvd",
    "fu_cvd",
    category,
    k = 20,
    times = 10
  )
  
  # Add the category name as a column to the result data frame
  result_df$Category <- category
  
  # Store the result data frame in the list
  results_list[[category]] <- result_df
}

# Combine all data frames into a single data frame
final_results_df <- do.call(rbind, results_list)

# Display the final combined results
print(final_results_df)

### ASCVD

In [ ]:
overall_calibration_slope(
    original_prevent_result,
    "ascvd_predicted_probability_10y",
    "ascvd",
    "fu_ascvd",
    k = 20,
    time = 10
  )

In [ ]:
# Initialize an empty list to store results
results_list <- list()

# Iterate over each category
for (category in categories) {
  # Calculate the stratified calibration slope for the current category
  result_df <- stratified_calibration_slope(
    original_prevent_result,
    "ascvd_predicted_probability_10y",
    "ascvd",
    "fu_ascvd",
    category,
    k = 20,
    times = 10
  )
  
  # Add the category name as a column to the result data frame
  result_df$Category <- category
  
  # Store the result data frame in the list
  results_list[[category]] <- result_df
}

# Combine all data frames into a single data frame
final_results_df <- do.call(rbind, results_list)

# Display the final combined results
print(final_results_df)

### HF

In [ ]:
overall_calibration_slope(
    original_prevent_result,
    "hf_predicted_probability_10y",
    "event_hf",
    "fu_hf",
    k = 20,
    time = 10
  )

In [ ]:
# Initialize an empty list to store results
results_list <- list()

# Iterate over each category
for (category in categories) {
  # Calculate the stratified calibration slope for the current category
  result_df <- stratified_calibration_slope(
    original_prevent_result,
    "hf_predicted_probability_10y",
    "event_hf",
    "fu_hf",
    category,
    k = 20,
    times = 10
  )
  
  # Add the category name as a column to the result data frame
  result_df$Category <- category
  
  # Store the result data frame in the list
  results_list[[category]] <- result_df
}

# Combine all data frames into a single data frame
final_results_df <- do.call(rbind, results_list)

# Display the final combined results
print(final_results_df)

## 5 year calibration 

### CVD

In [ ]:
overall_calibration_slope(
    original_prevent_result,
    "cvd_predicted_probability_5y",
    "cvd",
    "fu_cvd",
    k = 20,
    time = 5
  )

In [ ]:
# Initialize an empty list to store results
results_list <- list()

# Iterate over each category
for (category in categories) {
  # Calculate the stratified calibration slope for the current category
  result_df <- stratified_calibration_slope(
    original_prevent_result,
    "cvd_predicted_probability_5y",
    "cvd",
    "fu_cvd",
    category,
    k = 20,
    times = 5
  )
  
  # Add the category name as a column to the result data frame
  result_df$Category <- category
  
  # Store the result data frame in the list
  results_list[[category]] <- result_df
}

# Combine all data frames into a single data frame
final_results_df <- do.call(rbind, results_list)

# Display the final combined results
print(final_results_df)

### ASCVD

In [ ]:
overall_calibration_slope(
    original_prevent_result,
    "ascvd_predicted_probability_5y",
    "ascvd",
    "fu_ascvd",
    k = 20,
    time = 5
  )

In [ ]:
# Initialize an empty list to store results
results_list <- list()

# Iterate over each category
for (category in categories) {
  # Calculate the stratified calibration slope for the current category
  result_df <- stratified_calibration_slope(
    original_prevent_result,
    "ascvd_predicted_probability_5y",
    "ascvd",
    "fu_ascvd",
    category,
    k = 20,
    times = 5
  )
  
  # Add the category name as a column to the result data frame
  result_df$Category <- category
  
  # Store the result data frame in the list
  results_list[[category]] <- result_df
}

# Combine all data frames into a single data frame
final_results_df <- do.call(rbind, results_list)

# Display the final combined results
print(final_results_df)

### HF

In [ ]:
overall_calibration_slope(
    original_prevent_result,
    "hf_predicted_probability_5y",
    "event_hf",
    "fu_hf",
    k = 20,
    time = 5
  )

In [ ]:
# Initialize an empty list to store results
results_list <- list()

# Iterate over each category
for (category in categories) {
  # Calculate the stratified calibration slope for the current category
  result_df <- stratified_calibration_slope(
    original_prevent_result,
    "hf_predicted_probability_5y",
    "event_hf",
    "fu_hf",
    category,
    k = 20,
    times = 5
  )
  
  # Add the category name as a column to the result data frame
  result_df$Category <- category
  
  # Store the result data frame in the list
  results_list[[category]] <- result_df
}

# Combine all data frames into a single data frame
final_results_df <- do.call(rbind, results_list)

# Display the final combined results
print(final_results_df)

# Calculating PCE Risk

In [ ]:
original_pce_result <- analytic_raw_df %>%
  mutate(ascvd_predicted_probability_10y = case_when(
    # Formula for white women
    female == 1 & race == "White" ~ 
      1-(0.9665)^exp(-29.799 * log(age) +
      4.884 * log(age) * log(age) +
      13.540 * log(chol) -
      3.114 * log(age) * log(chol) -
      13.578 * log(hdlc) +
      3.149 * log(age) * log(hdlc) +
      2.019 * ifelse(antihtn == 1, log(sbp), 0) +
      1.957 * ifelse(antihtn == 0, log(sbp), 0) +
      7.574 * ifelse(cursmk == 1, 1, 0) -
      1.665 * log(age) * ifelse(cursmk == 1, 1, 0) +
      0.661 * ifelse(diabetes == 1, 1, 0) + 29.18),
    
    # Formula for black women
    female == 1 & race == "Black" ~ 
      1-(0.9533)^exp(17.114 * log(age) +
      0.94 * log(chol) -
      18.92 * log(hdlc) +
      4.475 * log(age) * log(hdlc) +
      29.291 * ifelse(antihtn == 1, log(sbp), 0) +
      27.82 * ifelse(antihtn == 0, log(sbp), 0) -
      6.432 * ifelse(antihtn == 1, log(age) * log(sbp), 0) -
      6.087 * ifelse(antihtn == 0, log(age) * log(sbp), 0) +
      0.691 * ifelse(cursmk == 1, 1, 0) +
      0.874 * ifelse(diabetes == 1, 1, 0) - 86.61),
    
    # Formula for white men
    female == 0 & race == "White" ~ 
      1-(0.9144)^exp(12.344 * log(age) +
      11.853 * log(chol) -
      2.664 * log(age) * log(chol) -
      7.99 * log(hdlc)+
      1.769 * log(age) * log(hdlc) +
      1.797 * ifelse(antihtn == 1, log(sbp), 0) +
      1.764 * ifelse(antihtn == 0, log(sbp), 0) +
      7.837 * ifelse(cursmk == 1, 1, 0) -
      1.795 * ifelse(cursmk == 1, log(age), 0)+
      0.658 * ifelse(diabetes == 1, 1, 0) - 61.18),
    
    # Formula for black men
    female == 0 & race == "Black" ~ 
      1-(0.8954)^exp(2.469 * log(age) +
      0.302 * log(chol) -
      0.307 * log(hdlc) +
      1.916 * ifelse(antihtn == 1, log(sbp), 0) +
      1.809 * ifelse(antihtn == 0, log(sbp), 0) +
      0.549 * ifelse(cursmk == 1, 1, 0) +
      0.645 * ifelse(diabetes == 1, 1, 0) - 19.54),
    
    # Default to NA for others
    TRUE ~ NA_real_
  ))


In [ ]:
original_pce_result <- original_pce_result[!is.na(original_pce_result$ascvd_predicted_probability_10y), ]

# Save predicted probability for better practice

In [ ]:
path = file.path("/PREVENT_PCE/Aim_1/original_pce_result")
save_artifacts_data(con, study, original_pce_result, path)
saved_path = file.path("/PREVENT_PCE/Aim_1/original_pce_result")
saved_df = load_artifacts_data(con, study, saved_path)
display_df(saved_df)

### For Aim 2

In [ ]:
pce_df <- original_pce_result %>% select(-ascvd_predicted_probability_10y)
path = file.path("/data/Luke_analysis_data/pce_df")
save_artifacts_data(con, study, pce_df, path)

# CI for PCE

In [ ]:
calculate_overall_concordance(
    data = original_pce_result,
    time_col = "fu_ascvd", 
    event_col = "ascvd", 
    prediction_col = "ascvd_predicted_probability_10y")

In [ ]:
# Loop through the categories and apply the function
for (var in categories) {
  cat("Calculating concordance for variable:", var, "\n")
  calculate_and_print_ci(
    data = original_pce_result,
    categorical_var = var,
    ci_function = calculate_concordance,
    time_col = "fu_ascvd", 
    event_col = "ascvd", 
    prediction_col = "ascvd_predicted_probability_10y"
  )
}

## C-Index (95% CI): 10 year PCE

In [ ]:
events <- c("ascvd")
time_horizons <- c(10)

categories <- c(
  "female", "ethnicity", "race", "cat_insurance", "cat_highest_edu_individual",
  "cat_med_nonadherence_risk", "cat_address_own_or_rent",
  "cat_current_address_dwelling_type", "cat_health_mgmt_nonmotivation_risk",
  "cat_highest_edu_household", "cat_healthcare_cost_risk", 
   "cat_address_change_60_months", 
  "cat_med_adherence_rate", "cat_current_address_median_income")


# 100 bootstrap, each = original size
nboot <- 100
N <- nrow(original_pce_result)

# store
results_list <- list()
res_counter <- 1

# loop over event and year
for (event in events) {
  time_col  <- paste0("fu_", event)
  event_col <- event
  for (horizon in time_horizons) {
    # pred probs
    pred_col <- paste0(event, "_predicted_probability_", horizon, "y")
    
    set.seed(123)
    for (b in seq_len(nboot)) {
      # resample
      sample_idx <- base::sample(seq_len(N), N, replace = TRUE)
      boot_data  <- original_pce_result[sample_idx, ]
      
      # for each subgroup category
      for (cat_var in categories) {
        
        # determine unique subgroup values
        unique_subgroups <- unique(boot_data[[cat_var]])
        
        for (subgroup_val in unique_subgroups) {
          
          # get c-index
          c_idx <- calculate_concordance(
            data           = boot_data,
            subgroup_col   = cat_var,
            subgroup_value = subgroup_val,
            time_col       = time_col,
            event_col      = event_col,
            prediction_col = pred_col
          )
          
          # store result
          results_list[[res_counter]] <- data.frame(
            event          = event,
            horizon        = horizon,
            category       = cat_var,
            subgroup_value = subgroup_val,
            replicate      = b,
            c_index        = c_idx
          )
          res_counter <- res_counter + 1
        }
      }
    }
  }
}

# concat results
df_results <- do.call(rbind, results_list)

# summarize
summary_results <- df_results %>%
  group_by(event, horizon, category, subgroup_value) %>%
  summarize(
    mean_c_index = mean(c_index, na.rm = TRUE),
    lower_ci     = quantile(c_index, probs = 0.025, na.rm = TRUE),
    upper_ci     = quantile(c_index, probs = 0.975, na.rm = TRUE),
    .groups      = "drop"
  )

# print
for (i in seq_len(nrow(summary_results))) {
  cat(summary_results$horizon[i], 'year', summary_results$event[i], 
    "| Category:", summary_results$category[i], 
    "| Subgroup:", summary_results$subgroup_value[i], 
    "| Mean C-Index (95%CI):", paste0(round(summary_results$mean_c_index[i], 4), "(", 
    round(summary_results$lower_ci[i], 4), "~", round(summary_results$upper_ci[i], 4), ")"), 
    "\n"
  )
}

## C-Index (95% CI): PREVENT - PCE (10 year ASCVD)

In [ ]:
# rename for clarity
original_pce_result <- original_pce_result %>%
  rename(pce_ascvd_predicted_probability_10y = ascvd_predicted_probability_10y)

# Transformation for PREVENT
all_df <- original_pce_result %>%
  mutate(
    year = as.numeric(format(index_dtt, "%Y")),
    egfr = 141.57 * (pmin(scre / 0.7, 1) ^ -0.241) * (pmax(scre / 0.7, 1) ^ -1.2) * (0.993839 ^ age) * 1.012,
    egfr = ifelse(female == 0, 
                  141.57 * (pmin(scre / 0.9, 1) ^ -0.302) * (pmax(scre / 0.9, 1) ^ -1.2) * (0.993839 ^ age),
                  egfr),
    fu_ascvd = pmin(fu_mi, fu_str, na.rm = TRUE),
    fu_cvd = pmin(fu_mi, fu_str, fu_hf, na.rm = TRUE),
    age = (age - 55) / 10,
    nonhdlc = (chol - hdlc) * 0.02586 - 3.5,
    hdlc = (hdlc * 0.02586 - 1.3) / 0.3,
    sbp1 = (pmin(sbp, 110) - 110) / 20,
    sbp2 = (pmax(sbp, 110) - 130) / 20,
    bmi1 = (pmin(bmi, 30) - 25) / 5,
    bmi2 = (pmax(bmi, 30) - 30) / 5,
    egfr1 = (pmin(egfr, 60) - 60) / -15,
    egfr2 = (pmax(egfr, 60) - 90) / -15,
    sbp2treat = sbp2 * antihtn,
    nonhdlctreat = nonhdlc * statin,
    agenonhdlc = age * nonhdlc,
    agehdlc = age * hdlc,
    agesbp2 = age * sbp2,
    agediabetes = age * diabetes,
    agecursmk = age * cursmk,
    agebmi2 = age * bmi2,
    ageegfr1 = age * egfr1,
    cons = 1
  )

# sex based index
male_idx   <- which(all_df$female == 0)
female_idx <- which(all_df$female == 1)

# Covariates for CVD and ASCVD
cvd_cov_male <- as.matrix(all_df[male_idx, c(
  "age","nonhdlc","hdlc","sbp1","sbp2","diabetes","cursmk","egfr1","egfr2",
  "antihtn","statin","sbp2treat","nonhdlctreat","agenonhdlc","agehdlc",
  "agesbp2","agediabetes","agecursmk","ageegfr1","cons"
)])

cvd_cov_female <- as.matrix(all_df[female_idx, c(
  "age","nonhdlc","hdlc","sbp1","sbp2","diabetes","cursmk","egfr1","egfr2",
  "antihtn","statin","sbp2treat","nonhdlctreat","agenonhdlc","agehdlc",
  "agesbp2","agediabetes","agecursmk","ageegfr1","cons"
)])

# 10 year ascvd
ascvd_pred_risk_male <- cvd_cov_male %*% ascvd_coef_male
ascvd_pred_prob_male_10y <- exp(ascvd_pred_risk_male) / (1 + exp(ascvd_pred_risk_male))
ascvd_pred_risk_female <- cvd_cov_female %*% ascvd_coef_female
ascvd_pred_prob_female_10y <- exp(ascvd_pred_risk_female) / (1 + exp(ascvd_pred_risk_female))

# Female PREVENT Subdataframe
all_df$prevent_ascvd_predicted_probability_10y <- NA
all_df$prevent_ascvd_predicted_probability_10y[male_idx]   <- ascvd_pred_prob_male_10y
all_df$prevent_ascvd_predicted_probability_10y[female_idx] <- ascvd_pred_prob_female_10y

diff_result <- all_df

In [ ]:
events <- c("ascvd")
time_horizons <- c(10)

categories <- c(
  "female", "ethnicity", "race", "cat_insurance", "cat_highest_edu_individual",
  "cat_med_nonadherence_risk", "cat_address_own_or_rent",
  "cat_current_address_dwelling_type", "cat_health_mgmt_nonmotivation_risk",
  "cat_highest_edu_household", "cat_healthcare_cost_risk", 
   "cat_address_change_60_months", 
  "cat_med_adherence_rate", "cat_current_address_median_income")


# 100 bootstrap, each = original size
nboot <- 100
N <- nrow(diff_result)

# store
results_list <- list()
res_counter <- 1

# loop over event and year
for (event in events) {
  time_col  <- paste0("fu_", event)
  event_col <- event
  for (horizon in time_horizons) {
    # pred probs
    prevent_pred_col <- paste0("prevent_", event, "_predicted_probability_", horizon, "y")
    pce_pred_col <- paste0("pce_", event, "_predicted_probability_", horizon, "y")
    
    set.seed(123)
    for (b in seq_len(nboot)) {
      # resample
      sample_idx <- base::sample(seq_len(N), N, replace = TRUE)
      boot_data  <- diff_result[sample_idx, ]
      
      # for each subgroup category
      for (cat_var in categories) {
        
        # determine unique subgroup values
        unique_subgroups <- unique(boot_data[[cat_var]])
        
        for (subgroup_val in unique_subgroups) {
          
          # get c-index
          prevent_c_idx <- calculate_concordance(
            data           = boot_data,
            subgroup_col   = cat_var,
            subgroup_value = subgroup_val,
            time_col       = time_col,
            event_col      = event_col,
            prediction_col = prevent_pred_col
          )

          pce_c_idx <- calculate_concordance(
            data           = boot_data,
            subgroup_col   = cat_var,
            subgroup_value = subgroup_val,
            time_col       = time_col,
            event_col      = event_col,
            prediction_col = pce_pred_col
          )
          
          # store result
          results_list[[res_counter]] <- data.frame(
            event          = event,
            horizon        = horizon,
            category       = cat_var,
            subgroup_value = subgroup_val,
            replicate      = b,
            c_index        = prevent_c_idx - pce_c_idx
          )
          res_counter <- res_counter + 1
        }
      }
    }
  }
}

# concat results
df_results <- do.call(rbind, results_list)

# summarize
summary_results <- df_results %>%
  group_by(event, horizon, category, subgroup_value) %>%
  summarize(
    mean_c_index = mean(c_index, na.rm = TRUE),
    lower_ci     = quantile(c_index, probs = 0.025, na.rm = TRUE),
    upper_ci     = quantile(c_index, probs = 0.975, na.rm = TRUE),
    .groups      = "drop"
  )

# print
for (i in seq_len(nrow(summary_results))) {
  cat(summary_results$horizon[i], 'year', summary_results$event[i], 
    "| Category:", summary_results$category[i], 
    "| Subgroup:", summary_results$subgroup_value[i], 
    "| Mean C-Index (95%CI):", paste0(round(summary_results$mean_c_index[i], 4), "(", 
    round(summary_results$lower_ci[i], 4), "~", round(summary_results$upper_ci[i], 4), ")"), 
    "\n"
  )
}

# Calibration for PCE

In [ ]:
overall_calibration_slope(
    original_pce_result,
    "ascvd_predicted_probability_10y",
    "ascvd",
    "fu_ascvd",
    k = 20,
    time = 10
  )

In [ ]:
results_list <- list()

# Iterate over each category
for (category in categories) {
  # Calculate the stratified calibration slope for the current category
  result_df <- stratified_calibration_slope(
    original_pce_result,
    "ascvd_predicted_probability_10y",
    "ascvd",
    "fu_ascvd",
    category,
    k = 20,
    times = 10
  )
  
  # Add the category name as a column to the result data frame
  result_df$Category <- category
  
  # Store the result data frame in the list
  results_list[[category]] <- result_df
}

# Combine all data frames into a single data frame
final_results_df <- do.call(rbind, results_list)

# Display the final combined results
print(final_results_df)

# Recalibrated PREVENT Model

## CVD

In [ ]:
recalibrate_survival <- function(data, time_var, event_var, linear_pred_var, timepoint = 10) {
  
  # Fit Kaplan-Meier survival curve
  km_fit <- survfit(Surv(data[[time_var]], data[[event_var]]) ~ 1, data = data)
  km_summary <- summary(km_fit, times = timepoint) 
  
  observed_survival <- km_summary$surv[1]  
  cat("Observed", timepoint, "-year Kaplan-Meier survival:", observed_survival, "\n")

  # Mean of the linear predictor
  mean_xbeta <- mean(data[[linear_pred_var]], na.rm = TRUE)  # Handle missing values
  data$xbc <- data[[linear_pred_var]] - mean_xbeta  # Centering for stability
  
  # Objective function to optimize
  objective_function <- function(S0) {
    predicted_survival <- mean(S0^exp(data$xbc))  # Mean predicted survival
    return(abs(predicted_survival - observed_survival))  # Minimize difference
  }
  
  # Optimize S0 using a bounded search (0 to 1)
  opt_result <- optimize(objective_function, interval = c(0, 1))
  S0 <- opt_result$minimum  # Extract optimal value
  
  cat("Recalibrated Baseline Survival at", timepoint, "Years (S0):", S0, "\n")
    
  # Calculate recalibrated survival
  recalibrated_col_name <- paste0("recalibrated_cvd_probability_", timepoint, "y")
  data[[recalibrated_col_name]] <- 1 - S0^exp(data$xbc)
    
  return(data)
}


In [ ]:
prevent_df_male = recalibrate_survival(
    prevent_df_male, 
    "fu_cvd", 
    "cvd", 
    "cvd_predicted_probability_10y", 
    timepoint = 10
    )

prevent_df_female = recalibrate_survival(
    prevent_df_female, 
    "fu_cvd", 
    "cvd", 
    "cvd_predicted_probability_10y", 
    timepoint = 10
    )
recalibrated_prevent_result <- rbind(prevent_df_female, prevent_df_male)

In [ ]:
print(1 - mean(prevent_df_male$recalibrated_cvd_probability_10y))
print(1 - mean(prevent_df_female$recalibrated_cvd_probability_10y))


In [ ]:
overall_calibration_slope(
    recalibrated_prevent_result,
    "recalibrated_cvd_probability_10y",
    "cvd",
    "fu_cvd",
    k = 30,
    time = 10 
  )

In [ ]:
km_fit <- survfit(Surv(fu_cvd, cvd) ~ 1, data = prevent_df)
surv_prob_10y <- summary(km_fit, times = 10)$surv
print(surv_prob_10y)

In [ ]:
expected = sum(recalibrated_prevent_result$recalibrated_cvd_probability_10y)/nrow(recalibrated_prevent_result)
observed = (1 - surv_prob_10y)  
print(expected/observed)

In [ ]:
expected = sum(original_prevent_result$cvd_predicted_probability_10y)/nrow(original_prevent_result)
observed = sum(1- surv_prob_10y)  
print(expected/observed)